# ДЗ-17 · Извлечение сущностей и событий из контрактов (NER + Information Extraction)

Извлекаем структурированную информацию из коммерческих контрактов **CUAD** с помощью локально развёрнутой инструктивной LLM в режиме zero/few-shot. Схема из 7 типов сущностей: `PERSON, ORG, MONEY, DATE, CONTRACT_TYPE, OBLIGATION, JURISDICTION`.

**Этапы (по заданию):**
1. Локальное развёртывание моделей (здесь — две маленькие модели Qwen2.5 на CPU; сравнение 7B quantized vs full — в `colab_7b.ipynb`).
2. Подготовка данных: подвыборка CUAD + промпты для извлечения.
3. Оптимизация: batch processing, измерение throughput.
4. Анализ: скорость (tokens/sec), качество (precision/recall/F1), ресурсы (RAM).

> ⚙️ **Где запускать.** Логика (парсинг, загрузка CUAD, метрики) работает везде. Сам inference требует рабочего `transformers`/`tokenizers`. На авторской Windows-машине связка `transformers 5.x / tokenizers 0.22` падает с access-violation, поэтому **inference-ячейки надёжнее всего запускать в Google Colab** (CPU или, лучше, T4 GPU). Загрузи туда `*.py` рядом с ноутбуком.

In [ ]:
# В Colab раскомментируй установку зависимостей:
# !pip -q install transformers datasets accelerate psutil pandas matplotlib gradio

# Чтобы ноутбук видел наши модули (ie_extractor.py, data_prep.py, evaluate.py, benchmark.py),
# держи их в той же папке. В Colab можно загрузить через files.upload() или git clone.
import sys, os
print('python', sys.version.split()[0])

## Этап 2 — Подготовка данных (CUAD)

`data_prep.get_subset()` грузит SQuAD-формат CUAD (parquet-ветка хаба), группирует строки по контракту и из человеческой разметки строит **gold** по тем типам, что есть в CUAD: `CONTRACT_TYPE` (Document Name), `ORG` (Parties), `DATE` (Agreement/Effective/Expiration Date), `JURISDICTION` (Governing Law). По `PERSON/MONEY/OBLIGATION` gold в CUAD нет — их модель извлекает для demo, но в метриках они не штрафуются.

In [ ]:
from data_prep import get_subset, gold_coverage, GOLD_ENTITY_TYPES

# 100–200 для demo; 500–1K для полноценного прогона (на CPU — долго).
samples = get_subset(n_docs=100, max_chars=2500)
print('контрактов:', len(samples))
print('типы с gold:', GOLD_ENTITY_TYPES)
print('покрытие gold (сумма спанов):', gold_coverage(samples))

In [ ]:
ex = samples[0]
print('TITLE:', ex.title)
print('TEXT[:400]:\n', ex.text[:400])
print('\nGOLD:', {k: v for k, v in ex.gold.items() if v})

## Этап 1 + 3 — Развёртывание модели и batch-извлечение

Оборачиваем инструктивную модель в `Extractor`. Промпт (`build_messages`) фиксирует строгий JSON-формат и даёт one-shot пример — это стабилизирует маленькие модели. `parse_entities` устойчиво вытаскивает JSON даже из «болтливого» ответа.

In [ ]:
from ie_extractor import Extractor, ENTITY_TYPES, build_messages

# посмотрим на промпт, который уйдёт в модель
for m in build_messages('Sample contract text ...'):
    print(f"[{m['role']}]\n{m['content'][:300]}\n")

In [ ]:
# Развёртываем модель локально (CPU, full precision fp32).
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # для скорости можно 'Qwen/Qwen2.5-0.5B-Instruct'
ext = Extractor(MODEL, device='cpu', dtype='float32',
                max_new_tokens=384, max_input_tokens=1536).load()

# Одиночное извлечение + статистика скорости
pred, stats = ext.extract_with_stats(samples[0].text)
print('STATS:', stats)
print('PRED :', {k: v for k, v in pred.items() if v})

In [ ]:
# Batch-обработка (этап 3): несколько документов за раз → выше throughput.
subset = samples[:20]                      # для demo берём 20
preds, bstats = ext.extract_batch([s.text for s in subset], batch_size=4)
print('batch stats:', bstats)

## Этап 4 — Анализ качества (precision / recall / F1)

Сравниваем предсказания с gold нечётко (нормализация + перекрытие токенов), считаем per-type и micro метрики только по типам, где есть gold.

In [ ]:
from evaluate import evaluate, format_report

metrics = evaluate(preds, [s.gold for s in subset])
print(format_report(metrics))

## Сравнение моделей и batch_size

`benchmark.benchmark_model` грузит модель один раз и гоняет на разных batch_size, возвращая скорость, качество и RAM. Сравним 0.5B и 1.5B.

In [ ]:
from benchmark import benchmark_model
import pandas as pd

bench_samples = samples[:20]
results = []
for m in ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']:
    results += benchmark_model(m, bench_samples, device='cpu', dtype='float32',
                               batch_sizes=[1, 4], max_new_tokens=384)

df = pd.DataFrame([{
    'model': r.model.split('/')[-1], 'batch': r.batch_size,
    'tok/s': r.tokens_per_sec, 'doc/s': r.docs_per_sec,
    'RAM_MB': r.ram_mb_peak, 'micro_F1': r.metrics.get('micro', {}).get('f1'),
} for r in results])
df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
piv_speed = df.pivot(index='model', columns='batch', values='tok/s')
piv_speed.plot(kind='bar', ax=ax[0], title='Throughput, tokens/sec'); ax[0].set_ylabel('tok/s')
df.groupby('model')['micro_F1'].max().plot(kind='bar', ax=ax[1], color='seagreen',
    title='Качество (micro F1)'); ax[1].set_ylabel('F1')
plt.tight_layout(); plt.show()

### Потребление ресурсов
`RAM_MB` в таблице — пиковая RSS процесса (psutil). На GPU `benchmark` дополнительно пишет `vram_mb_peak`. Ожидаемо: 1.5B заметно тяжелее 0.5B по RAM и медленнее по tokens/sec, но обычно точнее. Батчинг (batch>1) повышает docs/sec на CPU за счёт параллельной генерации.

## Demo-приложение

Интерактивная демонстрация — `app.py` (Gradio): вставляешь текст контракта, получаешь подсвеченные сущности + JSON.

```bash
python app.py                 # http://127.0.0.1:7860
IE_MODEL=Qwen/Qwen2.5-0.5B-Instruct python app.py   # быстрее
```

В Colab можно запустить с `demo.launch(share=True)`.

In [ ]:
# Быстрый inline-предпросмотр того, что показывает demo:
import app
txt = samples[0].text
ents = ext.extract(txt)
print('Сущности:', {k: v for k, v in ents.items() if v})
print('Сегменты подсветки:', app._to_highlighted(txt, ents)[:6])

## Выводы

* **LLM-as-extractor** позволяет извлекать 7 типов сущностей из юр-текстов без обучения отдельной NER-модели — достаточно промпта со строгой JSON-схемой и устойчивого парсера.
* **Gold из CUAD**: человеческую разметку 4 категорий удалось переиспользовать как gold, что даёт честный precision/recall без ручной разметки.
* **Размер vs скорость/качество**: 0.5B быстрее и легче по RAM, 1.5B точнее; батчинг повышает throughput на CPU.
* **7B и квантование** (4-bit vs full precision, VRAM, tokens/sec) — в `colab_7b.ipynb`.